# 🩻 Interactive Demo PGA-UNet (Prompt-Guided Attention U-Net)

This notebook reconstructs an interactive demo workflow for **PGA-UNet**, designed for Google Colab / Kaggle execution:

1. **Setup**: download the PGA-UNet codebase from GitHub, together with two test datasets (BTXRD, FracAtlas) and their corresponding checkpoints from Google Drive. A checkpoint trained by `pga-train-{128,256,512}.ipynb` can also be plugged in as an extra selectable checkpoint, downloaded from Google Drive by file ID (see `CUSTOM_CHECKPOINTS_GDRIVE` in Cell 3).
2. **Select image**: choose one test image with available ground truth. Because the six metrics require a reference mask, the notebook operates on labeled test images rather than arbitrary unlabeled uploads. Since there are **two checkpoints** (one trained per dataset), the interface also lets the user specify whether the image comes from **BTXRD or FracAtlas**. The default checkpoint matches the dataset, but the choice can be changed independently to probe cross-dataset generalization.
3. **Draw box**: click two points on the image to define a box, then press **Confirm**. The box is converted into a *prompt heatmap* (matched to the preprocessing in `dataset.py`), fed into PGA-UNet, and converted into a predicted mask.
4. **Compare**: compare the predicted mask against the ground truth and report all **six metrics**: Dice, IoU, Precision, Recall, HD95, and CBL, alongside the model's own no-GT confidence for that prompt (the CAD prompt-confidence gate, plus the QualityHead score when the loaded checkpoint has one).
5. **Continue / Finish**:
   - **Continue**: keep the same image, draw an additional prompt box, merge the new mask with the previously stored masks by **union**, and recompute the six metrics on the aggregated mask. This can be repeated over multiple rounds.
   - **Finish**: clear the current image, points, and masks, then return to image selection to start a new case.
6. **Suggest prompts** (optional): instead of drawing a box by hand, randomly probe the loaded image with a batch of boxes of varying size and position, rank them purely by the model's own no-GT confidence, and display the top 5 highest-scoring ones. Click any thumbnail to apply that exact box, exactly as if it had been drawn by hand and confirmed.

> **Scope note**: because the six metrics require ground truth, the demo operates only on **test** images with existing JSON annotations in the two datasets rather than on arbitrary external uploads.


**Cell 1: Environment setup + clone the PGA-UNet codebase**

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1: ENVIRONMENT SETUP (Colab / Kaggle)
# ══════════════════════════════════════════════════════
import os

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
os.chdir(BASE)

REPO_PGA = 'https://github.com/ThongLuc2k3/PGA_Unet2D.git'

if not os.path.exists(f'{BASE}/PGA_Unet2D'):
    !git clone -q {REPO_PGA} {BASE}/PGA_Unet2D
print('✅ Cloned the PGA-UNet codebase (`main`)')

!pip install -q gdown opencv-python matplotlib scikit-image gradio tqdm scipy

os.chdir(f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation')
print('✅ Setup complete, working directory:', os.getcwd())

# ── Google Drive download helper with fallback handling to avoid `FileURLRetrievalError` ──
# (anonymous `gdown` requests may be throttled by Google when a file is downloaded too often or not shared correctly as "Anyone with the link". The fallback uses the Drive API through the authenticated Colab account and does not depend on anonymous download limits.)
def robust_gdrive_download(file_id, output_path, quiet=False):
    import gdown
    if os.path.exists(output_path):
        return
    url = f'https://drive.google.com/uc?id={file_id}'
    try:
        gdown.download(url, output_path, quiet=quiet)
        if os.path.exists(output_path):
            return
    except Exception as e:
        print(f'  ⚠️ gdown failed ({e}); retrying with `fuzzy=True`...')
    try:
        gdown.download(id=file_id, output=output_path, quiet=quiet, fuzzy=True)
        if os.path.exists(output_path):
            return
    except Exception as e:
        print(f'  ⚠️ `gdown` with `fuzzy=True` also failed ({e}); trying the Drive API fallback (Colab account)...')
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        auth.authenticate_user()
        service = build('drive', 'v3')
        request = service.files().get_media(fileId=file_id)
        with open(output_path, 'wb') as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        print(f'  ✅ Downloaded successfully through the Drive API (Colab account): {output_path}')
        return
    except Exception as e:
        raise RuntimeError(
            f"❌ Unable to download the Google Drive file (id={file_id}).\n"
            f"Common causes:\n"
            f"  1. The file has not been shared 'Anyone with the link' (Viewer).\n"
            f"  2. The file has exceeded the public daily Google Drive download limit. Retry after a few hours,\n"
            f"     or download it manually via the browser at https://drive.google.com/uc?id={file_id}\n"
            f"     then upload it directly to {output_path}.\n"
            f"  3. The notebook is not running on Google Colab, so account-based authentication is unavailable for fallback step 3.\n"
            f"Original Drive API error: {e}"
        ) from e


**Cell 2: Download the two test datasets (BTXRD + FracAtlas) from Google Drive**

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 2: DOWNLOAD THE TWO TEST DATASETS (BTXRD + FracAtlas) FROM GOOGLE DRIVE
# ══════════════════════════════════════════════════════
import os

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'

# Google Drive file IDs for the two datasets (zip archives already containing train/val/test splits)
DATASETS_GDRIVE = {
    'BTXRD':     '1y7wD82n51hdLcbwrQwFtm-7sBnD_OJet',
    'FracAtlas': '1o4WUbs9faT6T-F-i59tPKSQNaROBY4DL',
}

for name, file_id in DATASETS_GDRIVE.items():
    ds_path = f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/dataset_{name}'
    if not os.path.exists(ds_path):
        zip_path = f'{BASE}/dataset_{name}.zip'
        robust_gdrive_download(file_id, zip_path, quiet=False)
        !unzip -oq {zip_path} -d {BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/
    print(f'✅ {name}: {ds_path}')


**Cell 3: Download the two PGA-UNet checkpoints (512x512), one per dataset**

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 3: DOWNLOAD THE PGA-UNET CHECKPOINTS (512x512) FROM GOOGLE DRIVE
# ══════════════════════════════════════════════════════
import os

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
CKPT_DIR = f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# Each dataset has its own PGA-UNet-512 checkpoint trained specifically on that dataset.
CHECKPOINTS_GDRIVE = {
    'BTXRD':     ('1NhRfMAf0T52-_5EqgqfdroRiHDk4A_m3', 'pga_btxrd_512_best.pth'),
    'FracAtlas': ('1lT9y4AJd4jLzDD0nmX2-rCmXgQItmFAY', 'pga_fracatlas_512_best.pth'),
}

CKPT_PATHS = {}
CKPT_USE_QUALITY_HEAD = {}
for name, (file_id, fname) in CHECKPOINTS_GDRIVE.items():
    fpath = os.path.join(CKPT_DIR, fname)
    robust_gdrive_download(file_id, fpath, quiet=False)
    CKPT_PATHS[name] = fpath
    CKPT_USE_QUALITY_HEAD[name] = False  # these two demo checkpoints predate QualityHead
    print(f'✅ {name}: {fpath}  ({os.path.getsize(fpath)//1024} KB)')

# ── Optional: plug in a checkpoint produced by `pga-train-{128,256,512}.ipynb`
# (the current center_mixed 80/20 protocol, plain Dice loss or a loss variant, see
# Prompt-Guided-XRay-Segmentation/README.md Step 2), downloaded from Google Drive by
# file ID (same mechanism as CHECKPOINTS_GDRIVE above). Each entry below becomes a
# selectable checkpoint in the "PGA-UNet checkpoint" dropdown, independent of which
# dataset the loaded image belongs to.
CUSTOM_CHECKPOINTS_GDRIVE = {
    # TODO_CHECKPOINT_ID: once a checkpoint from pga-train-{128,256,512}.ipynb is
    # trained and uploaded to Google Drive, add its file ID here, e.g.:
    # 'pga_center_mixed_x3_shift05_qhead': dict(
    #     file_id='TODO_CHECKPOINT_ID', fname='pga_unet_center_mixed_x3_shift05_qhead_512_best.pth',
    #     dataset='BTXRD', use_quality_head=True),
}
for name, cfg in CUSTOM_CHECKPOINTS_GDRIVE.items():
    fpath = os.path.join(CKPT_DIR, cfg['fname'])
    robust_gdrive_download(cfg['file_id'], fpath, quiet=False)
    CKPT_PATHS[name] = fpath
    CKPT_USE_QUALITY_HEAD[name] = cfg.get('use_quality_head', True)
    print(f"✅ {name}: {fpath}  (custom checkpoint, trained on {cfg['dataset']})")

**Cell 4: Load PGA-UNet and the shared processing utilities (preprocessing, heatmap generation, six metrics)**

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 4: LOAD THE MODEL AND SHARED PROCESSING UTILITIES
# ══════════════════════════════════════════════════════
import sys, json, cv2
import numpy as np
import torch
from scipy.ndimage import binary_erosion, distance_transform_edt

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
if f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation' not in sys.path:
    sys.path.insert(0, f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation')

from models.networks.prompt_unet_2D import PGA_UNet

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CANVAS = 512  # matches the resolution of the two checkpoints loaded in Cell 3

DATASET_DIRS = {
    'BTXRD':     dict(img_dir=f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/dataset_BTXRD/test/images',
                       json_dir=f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/dataset_BTXRD/test/annotations'),
    'FracAtlas': dict(img_dir=f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/dataset_FracAtlas/test/images',
                       json_dir=f'{BASE}/PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/dataset_FracAtlas/test/annotations'),
}

MODELS = {}
for name, ckpt_path in CKPT_PATHS.items():
    use_qhead = CKPT_USE_QUALITY_HEAD.get(name, False)
    m = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True, use_quality_head=use_qhead).to(DEVICE)
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    m.eval()
    MODELS[name] = m
    print(f'✅ PGA-UNet ({name}) ready: device={DEVICE}, use_quality_head={use_qhead}')

# ── Preprocessing: MUST match `Prompt-Guided-XRay-Segmentation/dataset.py` exactly ──
# (resize with aspect-ratio preservation followed by square padding, and use a plateau heatmap with a Gaussian kernel scaled to the target resolution)
def resize_and_pad(array, size, interpolation, pad_value=0):
    orig_h, orig_w = array.shape[:2]
    scale = min(size / orig_w, size / orig_h)
    new_w, new_h = max(1, int(round(orig_w * scale))), max(1, int(round(orig_h * scale)))
    resized = cv2.resize(array, (new_w, new_h), interpolation=interpolation)
    padded = np.full((size, size), pad_value, dtype=resized.dtype)
    pad_left, pad_top = (size - new_w) // 2, (size - new_h) // 2
    padded[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized
    return padded

def create_plateau_heatmap(bbox, size):
    # No minimum margin, fixed kernel=31 regardless of size: matches dataset.py's
    # PromptSegmentationDataset.create_plateau_heatmap exactly.
    heatmap = np.zeros((size, size), dtype=np.float32)
    x_min, y_min, x_max, y_max = bbox
    x_min, y_min = max(0, int(x_min)), max(0, int(y_min))
    x_max, y_max = min(size, int(x_max)), min(size, int(y_max))
    if x_max > x_min and y_max > y_min:
        heatmap[y_min:y_max, x_min:x_max] = 1.0
        heatmap = cv2.GaussianBlur(heatmap, (31, 31), 0)
    return heatmap

# ── Six metrics: computed exactly as in `Result/*/test-pga-*.ipynb` ──
def calc_hd95(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or not g.any(): return float(CANVAS)
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    return float(CANVAS) if not len(d1) or not len(d2) else float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_metrics_img(prob_np, gt_np, eps=1e-6):
    pm = (prob_np > 0.5).astype(np.float32)
    gm = (gt_np   > 0.5).astype(np.float32)
    tp = (pm * gm).sum(); fp = (pm * (1 - gm)).sum(); fn = ((1 - pm) * gm).sum()
    hd95 = calc_hd95(pm, gm)
    if gm.sum() == 0 or pm.sum() == 0:
        cbl = 0.0
    else:
        ys, xs = np.where(gm > 0.5); yp, xp = np.where(pm > 0.5)
        bbox_diag = np.sqrt((ys.max() - ys.min()) ** 2 + (xs.max() - xs.min()) ** 2) + eps
        cbl = float(np.clip(1. - np.sqrt((xp.mean() - xs.mean()) ** 2 + (yp.mean() - ys.mean()) ** 2) / bbox_diag, 0, 1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)), recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

METRIC_HDRS = ['Dice ↑', 'IoU ↑', 'Precision ↑', 'Recall ↑', 'HD95 ↓ (px)', 'CBL ↑']

# ── List test images with ground truth and load a 512x512 canvas ──
def list_test_images(dataset_key):
    dirs = DATASET_DIRS[dataset_key]
    names = []
    for fn in sorted(os.listdir(dirs['img_dir'])):
        if not fn.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        base = os.path.splitext(fn)[0]
        if os.path.exists(os.path.join(dirs['json_dir'], base + '.json')):
            names.append(fn)
    return names

def load_canvas(dataset_key, img_name):
    # Returns `(canvas_img uint8, canvas_gt float32 0/1)` in the exact 512x512 space seen by the model.
    dirs = DATASET_DIRS[dataset_key]
    base = os.path.splitext(img_name)[0]
    img = cv2.imread(os.path.join(dirs['img_dir'], img_name), cv2.IMREAD_GRAYSCALE)
    orig_h, orig_w = img.shape

    gt_native = np.zeros((orig_h, orig_w), dtype=np.uint8)
    with open(os.path.join(dirs['json_dir'], base + '.json'), 'r', encoding='utf-8') as f:
        data = json.load(f)
    for s in data.get('shapes', []):
        if s.get('shape_type') == 'polygon':
            cv2.fillPoly(gt_native, [np.array(s['points'], dtype=np.int32)], 1)

    canvas_img = resize_and_pad(img, CANVAS, cv2.INTER_LINEAR, pad_value=0)
    canvas_gt  = resize_and_pad(gt_native.astype(np.float32), CANVAS, cv2.INTER_NEAREST, pad_value=0.0)
    canvas_gt  = (canvas_gt > 0.5).astype(np.float32)
    return canvas_img, canvas_gt

def run_prompt(weight_key, canvas_img, bbox):
    # Maps a box in 512x512 canvas coordinates to a heatmap, runs PGA-UNet, and returns
    # a 512x512 binary mask plus the model's own no-GT confidence signals: `prompt_confidence`
    # (the CAD gate, always available) and `quality` (the QualityHead's estimate of its own
    # Dice, only when the checkpoint was trained with `use_quality_head=True`, else `None`).
    heatmap   = create_plateau_heatmap(bbox, CANVAS)
    img_t     = torch.from_numpy((canvas_img.astype(np.float32) / 255.0 - 0.5) / 0.5).unsqueeze(0).unsqueeze(0).to(DEVICE)
    heatmap_t = torch.from_numpy(heatmap).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    model     = MODELS[weight_key]
    use_qhead = CKPT_USE_QUALITY_HEAD.get(weight_key, False)
    with torch.no_grad():
        if use_qhead:
            logits, prompt_conf, quality = model(img_t, heatmap_t, return_confidence=True, return_quality=True)
        else:
            logits, prompt_conf = model(img_t, heatmap_t, return_confidence=True)
            quality = None
        prob = torch.sigmoid(logits)
    mask = (prob[0, 0].cpu().numpy() > 0.5).astype(np.float32)
    confidence = dict(prompt_confidence=float(prompt_conf.item()),
                       quality=float(quality.item()) if quality is not None else None)
    return mask, confidence

def suggest_prompts(weight_key, canvas_img, n_candidates=50, top_k=5,
                     size_frac_range=(0.15, 0.55), rng=None):
    # Randomly samples `n_candidates` box prompts of varying size and position over the
    # canvas (no spacing constraint between them), scores each purely from the model's
    # own no-GT confidence (QualityHead when the checkpoint has one, otherwise the CAD
    # prompt-confidence gate), and returns the `top_k` highest-scoring ones.
    if rng is None:
        rng = np.random.default_rng()
    use_qhead = CKPT_USE_QUALITY_HEAD.get(weight_key, False)

    candidates = []
    for _ in range(n_candidates):
        w = rng.uniform(*size_frac_range) * CANVAS
        h = rng.uniform(*size_frac_range) * CANVAS
        x_min = rng.uniform(0, CANVAS - w)
        y_min = rng.uniform(0, CANVAS - h)
        bbox = [x_min, y_min, x_min + w, y_min + h]
        mask, confidence = run_prompt(weight_key, canvas_img, bbox)
        score = confidence['quality'] if use_qhead else confidence['prompt_confidence']
        candidates.append(dict(bbox=bbox, mask=mask, confidence=confidence, score=score))

    candidates.sort(key=lambda c: c['score'], reverse=True)
    return candidates[:top_k]

def build_overlay(canvas_img_gray, acc_mask, canvas_gt, boxes, show_gt):
    base = cv2.cvtColor(canvas_img_gray, cv2.COLOR_GRAY2RGB).astype(np.float32)
    result = base.copy()
    if show_gt and canvas_gt is not None and canvas_gt.max() > 0:
        green = np.zeros_like(result); green[..., 1] = canvas_gt * 255
        result = cv2.addWeighted(result, 1.0, green, 0.35, 0)
    if acc_mask is not None and acc_mask.max() > 0:
        red = np.zeros_like(result); red[..., 0] = acc_mask * 255
        result = cv2.addWeighted(result, 1.0, red, 0.45, 0)
    result = np.clip(result, 0, 255).astype(np.uint8)
    for (bx1, by1, bx2, by2) in boxes:
        cv2.rectangle(result, (int(bx1), int(by1)), (int(bx2), int(by2)), (255, 255, 0), 1)
    return result

def metrics_to_markdown(m, round_idx, n_prompts, weight_key, confidence):
    # Note: bold markup (**...**) right inside an "###" heading line can render with
    # its first character clipped in some Gradio themes, so the checkpoint name here
    # stays plain text (the heading itself is already visually prominent).
    conf_line = f"No-GT confidence of the last prompt: CAD prompt-confidence **{confidence['prompt_confidence']:.1%}**"
    if confidence['quality'] is not None:
        conf_line += f", QualityHead confidence **{confidence['quality']:.1%}**"
    return (f"### 📊 Results after {n_prompts} prompts (round {round_idx}): PGA-UNet checkpoint: {weight_key}\n\n"
            f"{conf_line}\n\n"
            f"| {' | '.join(METRIC_HDRS)} |\n"
            f"|{'---|' * len(METRIC_HDRS)}\n"
            f"| {m['dice']:.4f} | {m['iou']:.4f} | {m['precision']:.4f} | "
            f"{m['recall']:.4f} | {m['hd95']:.2f} | {m['cbl']:.4f} |")

print('✅ Helper functions ready')

**Cell 5: Interactive Gradio interface (click two points -> box -> mask -> six metrics -> Continue / Finish)**

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 5: INTERACTIVE GRADIO INTERFACE
# ══════════════════════════════════════════════════════
import warnings
warnings.filterwarnings("ignore")
import gradio as gr

EMPTY_CANVAS_MSG = "⬅️ Select a dataset and a test image, then press 'Load image' to begin."


def on_dataset_change(dataset_key):
    names = list_test_images(dataset_key)
    img_update = gr.update(choices=names, value=names[0] if names else None)
    default_weight = dataset_key if dataset_key in MODELS else next(iter(MODELS))
    weight_update = gr.update(choices=list(MODELS.keys()), value=default_weight)
    return img_update, weight_update


def on_load_image(dataset_key, img_name):
    if not img_name:
        return (None, None, None, [], np.zeros((CANVAS, CANVAS), dtype=np.float32), [], 0,
                "⚠️ No image with ground truth was found.", None, "",
                gr.update(visible=False), gr.update(visible=False), None, "", [])
    canvas_img, canvas_gt = load_canvas(dataset_key, img_name)
    zero_mask = np.zeros((CANVAS, CANVAS), dtype=np.float32)
    display = build_overlay(canvas_img, zero_mask, canvas_gt, [], show_gt=False)
    status = "🎯 Click point 1, then click point 2 to create the box, then press 'Confirm'."
    return (display, canvas_img, canvas_gt, [], zero_mask, [], 0,
            status, None, "",
            gr.update(visible=False), gr.update(visible=False), None, "", [])


def _redraw_with_points(canvas_img, acc_mask, canvas_gt, boxes, points, show_gt):
    display = build_overlay(canvas_img, acc_mask, canvas_gt, boxes, show_gt)
    for p in points:
        cv2.circle(display, p, 6, (255, 60, 60), -1)
        cv2.circle(display, p, 7, (255, 255, 255), 1)
    if len(points) == 2:
        (px1, py1), (px2, py2) = points
        cv2.rectangle(display, (min(px1, px2), min(py1, py2)), (max(px1, px2), max(py1, py2)), (50, 220, 50), 2)
    return display


def on_click(evt: gr.SelectData, canvas_img, acc_mask, canvas_gt, boxes, points, show_gt):
    if canvas_img is None:
        return gr.update(), points, "⚠️ Please load an image first."
    pts = list(points)
    if len(pts) >= 2:
        pts = []
    pts.append((int(evt.index[0]), int(evt.index[1])))
    display = _redraw_with_points(canvas_img, acc_mask, canvas_gt, boxes, pts, show_gt)
    if len(pts) == 2:
        status = "🎯 The box is ready. Press 'Confirm' to segment."
    else:
        status = f"First corner captured: {pts[-1]}. Click the second corner to complete the box."
    return display, pts, status


def on_reset_points(canvas_img, acc_mask, canvas_gt, boxes, show_gt):
    if canvas_img is None:
        return gr.update(), [], "⚠️ Please load an image first."
    display = build_overlay(canvas_img, acc_mask, canvas_gt, boxes, show_gt)
    return display, [], "🔄 Points reset. Click two new points to draw another box."


def on_toggle_gt(canvas_img, acc_mask, canvas_gt, boxes, points, show_gt):
    if canvas_img is None:
        return gr.update()
    return _redraw_with_points(canvas_img, acc_mask, canvas_gt, boxes, points, show_gt)


def _apply_prompt_result(canvas_img, canvas_gt, acc_mask, boxes, weight_key, round_idx, show_gt,
                          bbox, mask, confidence):
    # Shared tail for both "draw two points -> Confirm" and "click a suggested prompt":
    # both end up with a box + its mask + its no-GT confidence, so both are folded into
    # the same accumulated mask, the same six metrics, and the same result display.
    acc_mask_new = np.maximum(acc_mask, mask)
    boxes_new = boxes + [tuple(bbox)]
    round_new = round_idx + 1

    m = calc_metrics_img(acc_mask_new, canvas_gt)
    result_display = build_overlay(canvas_img, acc_mask_new, canvas_gt, boxes_new, show_gt)
    md_text = metrics_to_markdown(m, round_new, len(boxes_new), weight_key, confidence)
    base_display = build_overlay(canvas_img, acc_mask_new, canvas_gt, boxes_new, show_gt)
    status = (f"✅ Segmentation round {round_new} completed. Choose 'Continue' to add another prompt or 'Finish' to switch to a different image.")

    return (result_display, md_text, acc_mask_new, boxes_new, round_new,
            base_display, [], status, gr.update(visible=True), gr.update(visible=True))


def on_confirm(points, canvas_img, canvas_gt, acc_mask, boxes, weight_key, round_idx, show_gt):
    if canvas_img is None:
        return (gr.update(), gr.update(), acc_mask, boxes, round_idx,
                gr.update(), points, "⚠️ Please load an image first.",
                gr.update(visible=False), gr.update(visible=False))
    if len(points) < 2:
        return (gr.update(), gr.update(), acc_mask, boxes, round_idx,
                gr.update(), points, "⚠️ You must click two points to create the box.",
                gr.update(visible=False), gr.update(visible=False))

    (x1, y1), (x2, y2) = points
    bbox = [min(x1, x2), min(y1, y2), max(x1, x2), max(y1, y2)]
    new_mask, confidence = run_prompt(weight_key, canvas_img, bbox)
    return _apply_prompt_result(canvas_img, canvas_gt, acc_mask, boxes, weight_key, round_idx, show_gt,
                                 bbox, new_mask, confidence)


def on_continue(canvas_img, acc_mask, canvas_gt, boxes, show_gt):
    display = build_overlay(canvas_img, acc_mask, canvas_gt, boxes, show_gt)
    status = "🎯 Click two points to draw an additional box prompt, then press 'Confirm'."
    return display, [], status, gr.update(visible=False), gr.update(visible=False)


def on_end(dataset_key):
    names = list_test_images(dataset_key)
    return (None, None, "", None, None, [], np.zeros((CANVAS, CANVAS), dtype=np.float32), [], 0,
            "⬅️ The current image and masks have been cleared. Select a new image, then press 'Load image'.",
            gr.update(visible=False), gr.update(visible=False),
            gr.update(choices=names, value=names[0] if names else None), None, "", [])


def on_suggest(canvas_img, weight_key):
    # Randomly probes the canvas with `n_candidates` box prompts of varying size and
    # position (no ground truth involved, no spacing constraint between candidates),
    # ranks them by the model's own no-GT confidence, and displays the top 5. The
    # returned list is also kept in `st_suggestions` so clicking a gallery thumbnail
    # can apply that exact box without re-running the model.
    if canvas_img is None:
        return None, "⚠️ Please load an image first.", []
    suggestions = suggest_prompts(weight_key, canvas_img)
    if not suggestions:
        return None, "⚠️ No suggested prompts could be generated; try again.", []

    gallery_items = []
    lines = [f"### 💡 Top {len(suggestions)} suggested prompts (checkpoint: {weight_key}), "
             f"ranked by the model's own no-GT confidence. Click a thumbnail to apply it, "
             f"exactly like drawing that box by hand and pressing Confirm.\n",
             "| Rank | Box (x1, y1, x2, y2) | CAD prompt-confidence | QualityHead confidence |",
             "|---|---|---|---|"]
    for i, c in enumerate(suggestions, start=1):
        bx1, by1, bx2, by2 = [int(round(v)) for v in c['bbox']]
        overlay = build_overlay(canvas_img, c['mask'], None, [c['bbox']], show_gt=False)
        cv2.putText(overlay, f"#{i}", (bx1 + 4, by1 + 22), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
        gallery_items.append((overlay, f"#{i}  score={c['score']:.1%}"))
        q_str = f"{c['confidence']['quality']:.1%}" if c['confidence']['quality'] is not None else "n/a"
        lines.append(f"| {i} | ({bx1}, {by1}, {bx2}, {by2}) | {c['confidence']['prompt_confidence']:.1%} | {q_str} |")
    return gallery_items, "\n".join(lines), suggestions


def on_select_suggestion(evt: gr.SelectData, suggestions, canvas_img, canvas_gt, acc_mask, boxes,
                          weight_key, round_idx, show_gt):
    # Applies the clicked suggestion exactly like Confirm does for a hand-drawn box: the
    # mask and confidence were already computed in on_suggest, so this just reuses them
    # rather than re-running the model.
    if canvas_img is None or not suggestions:
        return (gr.update(), gr.update(), acc_mask, boxes, round_idx,
                gr.update(), [], "⚠️ Please load an image and generate suggestions first.",
                gr.update(visible=False), gr.update(visible=False))
    chosen = suggestions[evt.index]
    return _apply_prompt_result(canvas_img, canvas_gt, acc_mask, boxes, weight_key, round_idx, show_gt,
                                 chosen['bbox'], chosen['mask'], chosen['confidence'])


with gr.Blocks(title="Interactive Demo PGA-UNet") as demo:
    gr.Markdown("""
    # 🩻 Interactive Demo: PGA-UNet (Prompt-Guided Attention U-Net)
    Select a dataset and a test image with available ground truth, click two points to draw a lesion box, inspect the segmentation output, and compare all six metrics against the reference mask. Multiple prompt rounds can be added with **Continue** before switching to another image with **Finish**. Use **Suggest prompts** to have the model rank a batch of randomly sized/positioned boxes by its own no-GT confidence, then click a thumbnail to apply it like any other box.
    """)

    st_canvas_img   = gr.State(None)
    st_canvas_gt    = gr.State(None)
    st_points       = gr.State([])
    st_acc_mask     = gr.State(np.zeros((CANVAS, CANVAS), dtype=np.float32))
    st_boxes        = gr.State([])
    st_round        = gr.State(0)
    st_suggestions  = gr.State([])

    with gr.Row():
        ds_dd     = gr.Dropdown(choices=list(DATASET_DIRS.keys()), value='BTXRD',
                                 label="1. Which dataset does this image belong to?")
        img_dd    = gr.Dropdown(choices=list_test_images('BTXRD'), label="2. Test image (with ground truth)")
        weight_dd = gr.Dropdown(choices=list(MODELS.keys()), value='BTXRD',
                                 label="3. PGA-UNet checkpoint used for inference")
        load_btn  = gr.Button("📥 Load image", variant="primary")

    status_tb = gr.Textbox(value=EMPTY_CANVAS_MSG, label="Status", interactive=False)

    with gr.Row():
        with gr.Column():
            canvas_display = gr.Image(label="3. 512x512 canvas: click two points to draw the box", type="numpy")
            with gr.Row():
                confirm_btn   = gr.Button("✅ Confirm Box", variant="primary")
                reset_pts_btn = gr.Button("↺ Reset points")
            show_gt_cb = gr.Checkbox(value=False, label="Show ground truth (green outline)")
        with gr.Column():
            result_display = gr.Image(label="4. Overlay result (red = prediction, green = ground truth)", type="numpy")
            metrics_md = gr.Markdown()
            with gr.Row():
                continue_btn = gr.Button("➕ Continue (add prompt)", visible=False)
                end_btn      = gr.Button("🏁 Finish (switch image)", visible=False, variant="stop")

    with gr.Row():
        suggest_btn = gr.Button("💡 Suggest prompts (random boxes, ranked by no-GT confidence)")
    suggest_gallery = gr.Gallery(label="Top-5 suggested prompts (click one to apply it)", columns=5, height=220)
    suggest_md = gr.Markdown()

    ds_dd.change(on_dataset_change, [ds_dd], [img_dd, weight_dd])

    load_btn.click(
        on_load_image, [ds_dd, img_dd],
        [canvas_display, st_canvas_img, st_canvas_gt, st_points, st_acc_mask, st_boxes, st_round,
         status_tb, result_display, metrics_md, continue_btn, end_btn, suggest_gallery, suggest_md, st_suggestions]
    )

    canvas_display.select(
        on_click, [st_canvas_img, st_acc_mask, st_canvas_gt, st_boxes, st_points, show_gt_cb],
        [canvas_display, st_points, status_tb]
    )

    reset_pts_btn.click(
        on_reset_points, [st_canvas_img, st_acc_mask, st_canvas_gt, st_boxes, show_gt_cb],
        [canvas_display, st_points, status_tb]
    )

    show_gt_cb.change(
        on_toggle_gt, [st_canvas_img, st_acc_mask, st_canvas_gt, st_boxes, st_points, show_gt_cb],
        [canvas_display]
    )

    confirm_btn.click(
        on_confirm, [st_points, st_canvas_img, st_canvas_gt, st_acc_mask, st_boxes, weight_dd, st_round, show_gt_cb],
        [result_display, metrics_md, st_acc_mask, st_boxes, st_round,
         canvas_display, st_points, status_tb, continue_btn, end_btn]
    )

    continue_btn.click(
        on_continue, [st_canvas_img, st_acc_mask, st_canvas_gt, st_boxes, show_gt_cb],
        [canvas_display, st_points, status_tb, continue_btn, end_btn]
    )

    end_btn.click(
        on_end, [ds_dd],
        [canvas_display, result_display, metrics_md, st_canvas_img, st_canvas_gt, st_points,
         st_acc_mask, st_boxes, st_round, status_tb, continue_btn, end_btn, img_dd,
         suggest_gallery, suggest_md, st_suggestions]
    )

    suggest_btn.click(
        on_suggest, [st_canvas_img, weight_dd],
        [suggest_gallery, suggest_md, st_suggestions]
    )

    suggest_gallery.select(
        on_select_suggestion,
        [st_suggestions, st_canvas_img, st_canvas_gt, st_acc_mask, st_boxes, weight_dd, st_round, show_gt_cb],
        [result_display, metrics_md, st_acc_mask, st_boxes, st_round,
         canvas_display, st_points, status_tb, continue_btn, end_btn]
    )

demo.launch(debug=True, share=True)